# Dentin Reference ROI Extraction and Intensity Normalization

This notebook extracts local tooth-root dentin reference ROIs, estimates a fixed reference dentin distribution from a predefined reference cohort, and applies affine dentin-based intensity normalization.

In [ ]:
from pathlib import Path
import pandas as pd

from src.dentin_reference_normalization import (
    DentinRoiConfig,
    estimate_reference_distribution,
    normalize_manifest,
)

In [ ]:
MANIFEST_CSV = Path('data/manifest.csv')
OUTPUT_DIR = Path('data/dentin_normalized_images')
STATS_DIR = Path('results/dentin_normalization')
STATS_DIR.mkdir(parents=True, exist_ok=True)

config = DentinRoiConfig(
    tooth_label=4,
    lesion_label=1,
    bone_label=3,
    material_label=2,
    margin_voxels=5,
    local_radius_voxels=35,
    material_buffer_voxels=2,
    min_voxels=100,
    clip_lower_percentile=1,
    clip_upper_percentile=99,
)

In [ ]:
manifest = pd.read_csv(MANIFEST_CSV)
reference_manifest = manifest[manifest['reference_cohort'].astype(int) == 1].copy()

reference, reference_case_stats, reference_failures = estimate_reference_distribution(
    reference_manifest,
    config=config,
    trim_proportion=0.10,
)

print(reference)
reference_case_stats.to_csv(STATS_DIR / 'reference_dentin_case_stats.csv', index=False)
reference_failures.to_csv(STATS_DIR / 'reference_dentin_failures.csv', index=False)

In [ ]:
normalized_stats, normalization_failures = normalize_manifest(
    manifest,
    reference=reference,
    output_dir=OUTPUT_DIR,
    config=config,
    suffix='_dentin_normalized',
)

normalized_stats.to_csv(STATS_DIR / 'dentin_normalization_stats.csv', index=False)
normalization_failures.to_csv(STATS_DIR / 'dentin_normalization_failures.csv', index=False)

normalized_stats.head()